# Deformable DETR 학습 노트북

**목적**: 표준 DETR(vera 실험 기준) 대비 Deformable DETR 성능 비교

**표준 DETR 베이스라인** (vera_tunning 최종):
- backbone_lr=1e-5, cls_lr=1e-4, weight_decay=1e-4
- mAP@50: 0.8565 / mAP@75:95: 0.8317

**Deformable DETR 개선 사항**:
1. Sparse attention (전체 이미지 대신 핵심 포인트에 집중)
2. Multi-scale feature 사용 (FPN-like)
3. Gradient clipping 추가 (max_norm=0.1)
4. Cosine LR Scheduler 추가
5. Val loss 기반 Early Stopping

In [ ]:
# ============================================================
# [Cell 0] 환경 세팅
# ============================================================
import os, sys

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/pill_detection_project'
    if not os.path.exists(REPO_DIR):
        os.system(f'git clone https://github.com/EuijeongHan/pill_detection_project.git {REPO_DIR}')
        os.system(f'cd {REPO_DIR} && git checkout detr-resolution')
    sys.path.insert(0, REPO_DIR)
    BASE_DIR = '/content/drive/MyDrive/data/초급_프로젝트/dataset'
    EXP_DIR  = '/content/drive/MyDrive/vera_detr/experiments'
else:
    BASE_DIR = '../../../data'
    EXP_DIR  = '../../../results/detr'

os.makedirs(EXP_DIR, exist_ok=True)
VAL_JSON = f'{BASE_DIR}/val_letterbox.json'

print(f'✅ BASE_DIR : {BASE_DIR}')
print(f'✅ EXP_DIR  : {EXP_DIR}')

In [ ]:
# ============================================================
# [Cell 1] 패키지 설치
# ============================================================
!pip install transformers timm pycocotools torchmetrics -q

In [ ]:
# ============================================================
# [Cell 2] DataLoader
# ============================================================
from src.models.detr.dataset import get_deformable_loaders

train_loader, val_loader, idx2cat = get_deformable_loaders(
    base_dir=BASE_DIR,
    batch_size=4,
    num_workers=2,
)
NUM_CLASSES = len(idx2cat)  # 73
print(f'✅ num_classes : {NUM_CLASSES}')

In [ ]:
# ============================================================
# [Cell 3] 팀 평가 모듈
# ============================================================
from src.evaluation import (
    evaluate_all,
    init_history,
    update_history,
    save_history,
    load_history,
    plot_training_history,
    plot_compare_histories,
)
print('✅ 팀 평가 모듈 로드 완료')

In [ ]:
# ============================================================
# [Cell 4] 모델 로드
# ============================================================
import torch
from transformers import DeformableDetrForObjectDetection

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ device: {device}')

model = DeformableDetrForObjectDetection.from_pretrained(
    'SenseTime/deformable-detr',
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,
)
model.to(device)
print('✅ Deformable DETR 로드 완료')

# 경고 설명:
# - num_batches_tracked → BatchNorm 통계, 무시해도 됨
# - class_labels_classifier shape mismatch → COCO 91클래스 → 우리 73클래스, 의도된 동작

In [ ]:
# ============================================================
# [Cell 5] Optimizer + LR Scheduler
# ============================================================
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS       = 30
BACKBONE_LR  = 1e-5
CLS_LR       = 1e-4
WEIGHT_DECAY = 1e-4

# DeformableDetr은 class_labels_classifier 속성이 없음
# backbone(feature extractor) vs head(encoder/decoder/embed)으로 분리
backbone_params = [p for n, p in model.named_parameters() if 'backbone' in n]
head_params     = [p for n, p in model.named_parameters() if 'backbone' not in n]

optimizer = AdamW([
    {'params': backbone_params, 'lr': BACKBONE_LR},
    {'params': head_params,     'lr': CLS_LR},
], weight_decay=WEIGHT_DECAY)

scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)

print(f'✅ backbone params : {len(backbone_params)}개  lr={BACKBONE_LR}')
print(f'✅ head params     : {len(head_params)}개  lr={CLS_LR}')
print(f'✅ Scheduler       : CosineAnnealingLR (T_max={EPOCHS})')

In [ ]:
# ============================================================
# [Cell 6] 학습 / 검증 함수
# ============================================================

def train_one_epoch(model, loader, optimizer, device, max_norm=0.1):
    model.train()
    total_loss = 0
    for encoding, _ in loader:
        pixel_values = encoding['pixel_values'].to(device)
        pixel_mask   = encoding['pixel_mask'].to(device)
        labels       = [{k: v.to(device) for k, v in t.items()}
                        for t in encoding['labels']]

        outputs = model(pixel_values=pixel_values,
                        pixel_mask=pixel_mask,
                        labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        # Gradient Clipping — DETR 계열 학습 안정화 핵심
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)


def eval_one_epoch(model, loader, device):
    """Val loss 계산 (Early Stopping 기준)"""
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for encoding, _ in loader:
            pixel_values = encoding['pixel_values'].to(device)
            pixel_mask   = encoding['pixel_mask'].to(device)
            labels       = [{k: v.to(device) for k, v in t.items()}
                            for t in encoding['labels']]
            outputs = model(pixel_values=pixel_values,
                            pixel_mask=pixel_mask,
                            labels=labels)
            total_loss += outputs.loss.item()
    return total_loss / len(loader)


def convert_outputs(model, loader, device, idx2cat, score_threshold=0.001):
    """Deformable DETR 출력 → evaluate_all 포맷 변환"""
    predictions = []
    model.eval()
    with torch.no_grad():
        for encoding, raw_targets in loader:
            pixel_values = encoding['pixel_values'].to(device)
            pixel_mask   = encoding['pixel_mask'].to(device)
            outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask)

            for i, raw_t in enumerate(raw_targets):
                image_id   = raw_t['image_id']
                W, H       = 800, 800
                logits     = outputs.logits[i]
                boxes      = outputs.pred_boxes[i]
                scores_all = logits.softmax(-1)
                scores, labels = scores_all[:, :-1].max(-1)

                for score, label, box in zip(scores, labels, boxes):
                    score = score.item()
                    if score < score_threshold:
                        continue
                    cx, cy, bw, bh = box.tolist()
                    predictions.append({
                        'image_id':    image_id,
                        'category_id': idx2cat.get(label.item(), label.item()),
                        'bbox_xyxy': [
                            (cx - bw/2) * W, (cy - bh/2) * H,
                            (cx + bw/2) * W, (cy + bh/2) * H,
                        ],
                        'score': score,
                    })
    return predictions


class EarlyStopping:
    """Val loss 기반 Early Stopping (표준 DETR은 train loss 기준이었던 문제 수정)"""
    def __init__(self, patience=7, save_path=None):
        self.patience   = patience
        self.save_path  = save_path
        self.counter    = 0
        self.best_loss  = float('inf')
        self.stop       = False

    def step(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter   = 0
            if self.save_path:
                torch.save(model.state_dict(), self.save_path)
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop

print('✅ 학습/평가 함수 준비 완료')

In [ ]:
# ============================================================
# [Cell 7] 학습 실행
# ============================================================
import json

EXP_NAME  = f'deformable_backbone{BACKBONE_LR}_cls{CLS_LR}_wd{WEIGHT_DECAY}'
SAVE_PATH = f'{EXP_DIR}/{EXP_NAME}_best.pth'

early_stopping = EarlyStopping(patience=7, save_path=SAVE_PATH)
train_losses, val_losses = [], []

print(f'🚀 학습 시작: {EXP_NAME}')
print(f'   Gradient Clipping : max_norm=0.1')
print(f'   LR Scheduler      : CosineAnnealingLR')
print(f'   Early Stopping    : val loss 기준, patience=7\n')

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_loss   = eval_one_epoch(model, val_loader, device)
    scheduler.step()

    train_losses.append(round(train_loss, 4))
    val_losses.append(round(val_loss, 4))

    print(f'Epoch [{epoch:02d}/{EPOCHS}]  train: {train_loss:.4f}  val: {val_loss:.4f}  '
          f'lr: {scheduler.get_last_lr()[0]:.2e}')

    if early_stopping.step(val_loss, model):
        print(f'\n⏹️  Early Stopping @ epoch {epoch}  (best val: {early_stopping.best_loss:.4f})')
        break

# train log 저장
log = {
    'config': {'backbone_lr': BACKBONE_LR, 'cls_lr': CLS_LR,
               'weight_decay': WEIGHT_DECAY, 'epochs': EPOCHS,
               'grad_clip': 0.1, 'scheduler': 'CosineAnnealingLR',
               'early_stopping': 'val_loss'},
    'train_losses': train_losses,
    'val_losses':   val_losses,
    'stopped_epoch': epoch,
}
with open(f'{EXP_DIR}/{EXP_NAME}_train_log.json', 'w') as f:
    json.dump(log, f, indent=2)

print(f'\n✅ 학습 완료 | best 가중치: {SAVE_PATH}')

In [ ]:
# ============================================================
# [Cell 8] mAP 평가
# ============================================================

# best 가중치 로드
model.load_state_dict(torch.load(SAVE_PATH, map_location=device))

val_predictions = convert_outputs(model, val_loader, device, idx2cat)

metrics = evaluate_all(
    gt_json_path=VAL_JSON,
    predictions=val_predictions,
    conf_threshold=0.25,
    pr_iou_threshold=0.5,
    temp_json_path=f'deformable_temp_{EXP_NAME}.json',
)

print(f'\n====== Deformable DETR 결과 ======')
print(f'  mAP@50    : {metrics["mAP@50"]:.4f}  (표준 DETR 베이스라인: 0.8565)')
print(f'  mAP@75:95 : {metrics["mAP@75:95"]:.4f}  (표준 DETR 베이스라인: 0.8317)')
print(f'  precision : {metrics["precision"]:.4f}')
print(f'  recall    : {metrics["recall"]:.4f}')

In [ ]:
# ============================================================
# [Cell 9] History 저장 + 시각화
# ============================================================

history = init_history()
for ep, (tl, vl) in enumerate(zip(train_losses, val_losses), 1):
    update_history(history, epoch=ep, train_loss=tl, val_loss=vl, metrics=None)

history['mAP@50'][-1]    = metrics['mAP@50']
history['mAP@75:95'][-1] = metrics['mAP@75:95']
history['precision'][-1] = metrics['precision']
history['recall'][-1]    = metrics['recall']

save_history(history, f'{EXP_DIR}/history_{EXP_NAME}.json')
plot_training_history(history, title_prefix='Deformable DETR')

print('✅ History 저장 완료')

In [ ]:
# ============================================================
# [Cell 10] 표준 DETR vs Deformable DETR 비교 요약
# ============================================================

baseline = {
    'model':      '표준 DETR',
    'mAP@50':     0.8565,
    'mAP@75:95':  0.8317,
    'grad_clip':  '없음',
    'scheduler':  '없음',
    'es_basis':   'train loss',
}
ours = {
    'model':      'Deformable DETR',
    'mAP@50':     metrics['mAP@50'],
    'mAP@75:95':  metrics['mAP@75:95'],
    'grad_clip':  'max_norm=0.1',
    'scheduler':  'CosineAnnealingLR',
    'es_basis':   'val loss',
}

print('\n' + '='*65)
print(f'{"":20} {"표준 DETR":>15} {"Deformable DETR":>15}')
print('='*65)
for key in ['mAP@50', 'mAP@75:95', 'grad_clip', 'scheduler', 'es_basis']:
    bv = baseline[key]
    ov = ours[key]
    b_str = f'{bv:.4f}' if isinstance(bv, float) else bv
    o_str = f'{ov:.4f}' if isinstance(ov, float) else ov
    print(f'{key:<20} {b_str:>15} {o_str:>15}')
print('='*65)

diff_50    = metrics['mAP@50']    - 0.8565
diff_75_95 = metrics['mAP@75:95'] - 0.8317
print(f'\n▲ mAP@50    변화: {diff_50:+.4f}')
print(f'▲ mAP@75:95 변화: {diff_75_95:+.4f}')